# Built-in middleware

일반적인 에이전트 사용 사례를 위한 사전 구축된 미들웨어

> https://docs.langchain.com/oss/python/langchain/middleware/overview

> https://docs.langchain.com/oss/python/langchain/middleware/built-in

> https://reference.langchain.com/python/langchain/middleware/

### LLM tool emulator (LLM 도구 모방기)

- 실제 도구를 실행하지 않고 에이전트 동작을 테스트합니다.
- 외부 도구를 사용할 수 없거나 비용이 많이 드는 경우 에이전트를 개발합니다.
- 실제 도구를 구현하기 전에 에이전트 워크플로 프로토타입을 제작합니다.

> https://docs.langchain.com/oss/python/langchain/middleware/built-in#llm-tool-emulator

In [21]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [22]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """특정 위치의 현재 날씨 정보를 가져옵니다."""
    return f"{location}의 날씨"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """이메일을 발송합니다."""
    return "이메일 발송 완료"

In [23]:
# 모듈 설치 요망
# uv add langchain_anthropic

from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator

# Use custom model for emulation
agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator(model="google_genai:gemini-2.5-flash-lite")],
)

In [24]:
agent.invoke({"messages": [{"role": "user", "content": "부산 날씨가 어때?"}]})

{'messages': [HumanMessage(content='부산 날씨가 어때?', additional_kwargs={}, response_metadata={}, id='294da76c-2d22-4f31-bca6-b5bbc0cf2b90'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "\\ubd80\\uc0b0"}'}, '__gemini_function_call_thought_signatures__': {'87e55be6-66f3-49a0-b377-5c8b53f284af': 'CrQBAb4+9vtHrayF1+jIdGQU1brJCRFoAiz2BSIQR4IcN/IwAn7pBxFzLHF0qT+dCp1JGtRzXMoRkfGAFQDt6KlLHE46PUa8jXdXalRRlyOpdq+wn+vs5Natj9K8HgoFcVREpjhkjzlPWCvb2AkQuggAxIlAhg1thlPCwPv8WaR6UeSn0xv7xbs9BHqC9H7JFhIkYJ/iACaqWbKEFyNQeWBMRZPq5vhqJdu6OT/kmkMMns4NAQ85'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cfc0a-9728-7d92-837a-55b9466e678e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': '부산'}, 'id': '87e55be6-66f3-49a0-b377-5c8b53f284af', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 113, 'output_toke

In [25]:
agent.invoke({"messages": [{"role": "user", "content": "김일남님께 이메일 발송해줘(제목: 랭체인 미들웨어 공부를 하세요, 내용: 랭체인 미들웨어 공부를 공식 문서를 보고 집중적으로 하도록 격려하는 내용)"}]})

{'messages': [HumanMessage(content='김일남님께 이메일 발송해줘(제목: 랭체인 미들웨어 공부를 하세요, 내용: 랭체인 미들웨어 공부를 공식 문서를 보고 집중적으로 하도록 격려하는 내용)', additional_kwargs={}, response_metadata={}, id='3155cfd7-8711-4726-a73f-becd346c5b47'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email', 'arguments': '{"to": "\\uae40\\uc77c\\ub0a8", "subject": "\\ub7ad\\uccb4\\uc778 \\ubbf8\\ub4e4\\uc6e8\\uc5b4 \\uacf5\\ubd80\\ub97c \\ud558\\uc138\\uc694", "body": "\\ub7ad\\uccb4\\uc778 \\ubbf8\\ub4e4\\uc6e8\\uc5b4 \\uacf5\\ubd80\\ub97c \\uacf5\\uc2dd \\ubb38\\uc11c\\ub97c \\ubcf4\\uace0 \\uc9d1\\uc911\\uc801\\uc73c\\ub85c \\ud558\\ub3c4\\ub85d \\uaca9\\ub824\\ud558\\ub294 \\ub0b4\\uc6a9"}'}, '__gemini_function_call_thought_signatures__': {'5f67796b-6e1b-4a10-bdf8-a21239ca9ea9': 'CuMEAb4+9vvrS7qtfWcWx2CPnTJNMv+1J1h7VFu0hbFWgMaYVwCHtiSypDFQZTt9zJwad84aV3Vux8VsQz943TsHkjfQB/J/Nf1dZ17DGmxNT68vCSS2KuoiSoKofTCcjKtkRJsazChnVAL1xGdALvVfxY7J0DcGy+TF0Ns+zx9rXFkopLkqj3DBNwfVrbP3IxiJFzMoghK0lVUqjGy2WPKc3sr1n3F

In [26]:
# Emulate specific tools only
agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator(model="google_genai:gemini-2.5-flash-lite", tools=["get_weather"])],
)

In [11]:
agent.invoke({"messages": [{"role": "user", "content": "부산 날씨가 어때?"}]})

{'messages': [HumanMessage(content='부산 날씨가 어때?', additional_kwargs={}, response_metadata={}, id='30fd068e-4220-45f9-80ad-5c12cf810a45'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "\\ubd80\\uc0b0"}'}, '__gemini_function_call_thought_signatures__': {'84f857a2-f6e7-47d0-a411-a63205d60959': 'CrQBAb4+9vsfVdSU8fmCD5sbm0PkS6A3vh1/FmgwiacdiI4FauOuybP0R/Ue/G6AlTxb+kKgIIKo5Umnwndn8zIwBGVDYnF8Y9m9nmBQBlh1yc3TLo4CF+xczWRuWLuZ/GepmyBwoMneTz51GmWVoxsEUPCEv7u7LcieXQCVup9lCGWmHUlOKEMrZSoqT3hMXk1XQPgOGTt4f84rcvFSKd6ogE/6PldLT2XyK47qGNO45h8s6YRH'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cfc06-2510-7461-a8ee-0dd676dee475-0', tool_calls=[{'name': 'get_weather', 'args': {'location': '부산'}, 'id': '84f857a2-f6e7-47d0-a411-a63205d60959', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 113, 'output_toke

In [27]:
agent.invoke({"messages": [{"role": "user", "content": "김일남님께 이메일 발송해줘(제목: 랭체인 미들웨어 공부를 하세요, 내용: 랭체인 미들웨어 공부를 공식 문서를 보고 집중적으로 하도록 격려하는 내용)"}]})

{'messages': [HumanMessage(content='김일남님께 이메일 발송해줘(제목: 랭체인 미들웨어 공부를 하세요, 내용: 랭체인 미들웨어 공부를 공식 문서를 보고 집중적으로 하도록 격려하는 내용)', additional_kwargs={}, response_metadata={}, id='de6d4498-0656-46d3-9df2-8b00ce248145'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email', 'arguments': '{"body": "\\ub7ad\\uccb4\\uc778 \\ubbf8\\ub4e4\\uc6e8\\uc5b4 \\uacf5\\ubd80\\ub97c \\uacf5\\uc2dd \\ubb38\\uc11c\\ub97c \\ubcf4\\uace0 \\uc9d1\\uc911\\uc801\\uc73c\\ub85c \\ud558\\ub3c4\\ub85d \\uaca9\\ub824\\ud558\\ub294 \\ub0b4\\uc6a9", "to": "\\uae40\\uc77c\\ub0a8\\ub2d8", "subject": "\\ub7ad\\uccb4\\uc778 \\ubbf8\\ub4e4\\uc6e8\\uc5b4 \\uacf5\\ubd80\\ub97c \\ud558\\uc138\\uc694"}'}, '__gemini_function_call_thought_signatures__': {'60638943-3e6f-47e3-bf98-9d936f4a9205': 'Ct4EAb4+9vs8+XD/LH+Y9UifVdUbhcBM4tlStK9J0VcsE4Q8VM/mxlUC7ZksbrBYbZRHruC6cQ8DhuIBJyHUaH2cLPI9NGqJ4hwCMGflrtST8c5NukM01IjtO6qAKyGsXAM23aGTzbTbSGPIloZXr4ckfEhQZqdwMuktoYPkD761wmpgkAARoB8ejrguJOqm5tKAocKexc2bnwtPRUUcuSbQ